# 04. Structured Output

## What you'll learn

- How to prompt an LLM to return valid JSON instead of free text
- How to parse JSON from messy LLM output (code fences, surrounding text)
- How to validate parsed data against an expected schema
- How to build a retry loop that feeds errors back to the LLM for self-correction
- How to compose all of the above into a reusable extraction pipeline

## Prerequisites

- [01. Hello LLM](01_hello_llm.ipynb) — making API calls with the `chat` helper
- [02. Basic Agent Loop](02_basic_agent_loop.ipynb) — conversation history and multi-turn interaction
- [Appendix 04. Strings and JSON](../appendix/04_strings_and_json.ipynb) — `json.loads`, `json.dumps`, regex extraction
- [Appendix 05. Error Handling](../appendix/05_error_handling.ipynb) — `try`/`except`, retry patterns

## Why this matters

LLMs return **free text**. Agents need **structured data** — tool calls with specific arguments, extracted entities with known fields, formatted reports with consistent shapes. The gap between "the LLM *usually* returns valid JSON" and "the LLM *always* returns valid JSON" is exactly the gap that breaks agents in production. A single malformed response crashes your agent loop.

This notebook builds the parsing, validation, and retry infrastructure that makes structured LLM output **reliable**. Every technique here feeds directly into the tool-use and ReAct patterns in upcoming notebooks.

> **Further reading:**
> - [OpenAI — Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) — how OpenAI approaches the same problem at the API level
> - [JSON Schema — Getting Started](https://json-schema.org/learn/getting-started-step-by-step) — the formal spec behind structured output validation
> - [Outlines (dottxt-ai)](https://github.com/dottxt-ai/outlines) — constrained generation that guarantees valid JSON at the token level

---

## 1. The Problem

Ask an LLM a question and you get a natural-language answer. That's great for chatbots, but agents need to **do things** with the output: look up a key, pass arguments to a function, store data in a database.

Let's see what happens when we naively ask for JSON.

In [ ]:
import sys
sys.path.insert(0, "..")
from utils import chat

In [ ]:
# A naive request for JSON
response = chat(
    [{"role": "user", "content": "Give me info about Marie Curie as JSON."}],
    temperature=0,
)
print(response)

You'll likely see something like:

```
Sure! Here's the information about Marie Curie in JSON format:

```json
{"name": "Marie Curie", ...}
```

The response might look like JSON *somewhere in there*, but it's wrapped in explanation text, code fences, or both. Calling `json.loads()` on this will crash.

The rest of this notebook solves this problem in layers:
1. **Better prompting** — tell the LLM exactly what format we need
2. **Robust parsing** — extract JSON even from messy output
3. **Schema validation** — verify the parsed data has the right shape
4. **Retry with feedback** — when it fails, tell the LLM what went wrong and try again

---

## 2. JSON Prompting

The single most effective technique: **show the LLM the exact schema you expect** and tell it to return *only* JSON. No preamble, no explanation — just the data.

Key prompting strategies:
- Include the word "ONLY" — "Respond ONLY with valid JSON"
- Show the exact shape with placeholder values
- Specify types explicitly ("number", "string", etc.)
- Use `temperature=0` for maximum consistency

In [ ]:
schema_prompt = """Extract the following from the text and respond ONLY with valid JSON:
{
  "name": "person's name",
  "age": number,
  "occupation": "their job"
}

Text: Marie Curie was a physicist born in 1867. She won two Nobel Prizes."""

response = chat([{"role": "user", "content": schema_prompt}], temperature=0)
print(response)

This is much better — most of the time the LLM returns clean JSON. But "most of the time" isn't good enough for an agent. The LLM might still:

- Wrap the JSON in markdown code fences (` ```json ... ``` `)
- Add a brief explanation before or after the JSON
- Return slightly different keys than requested
- Use wrong types (a string `"159"` instead of the number `159`)

We need a parsing layer that handles all of this.

---

## 3. Parsing with `json.loads()`

The goal: take whatever the LLM returned and extract a Python dict from it. We build `safe_parse_json()` to handle the most common LLM output patterns:

1. Clean JSON (just parse it)
2. JSON wrapped in markdown code fences (strip the fences)
3. JSON embedded in surrounding text (find the `{...}` block)

This is the same pattern from [Appendix 04](../appendix/04_strings_and_json.ipynb), extended to be more robust.

In [ ]:
import json
import re


def safe_parse_json(text):
    """Try to parse JSON from LLM output, handling common issues."""
    text = text.strip()

    # Remove markdown code fences
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # Try direct parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to find a JSON object in the text
    json_match = re.search(r"\{.*\}", text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            pass

    return None

Let's test it with the various output formats LLMs actually produce:

In [ ]:
# Test case 1: Clean JSON
clean = '{"name": "Marie Curie", "age": 159, "occupation": "physicist"}'
print("Clean JSON:", safe_parse_json(clean))

# Test case 2: JSON in markdown code fences
fenced = '```json\n{"name": "Marie Curie", "age": 159, "occupation": "physicist"}\n```'
print("Fenced JSON:", safe_parse_json(fenced))

# Test case 3: JSON with surrounding text
messy = 'Here is the extracted data:\n{"name": "Marie Curie", "age": 159, "occupation": "physicist"}\nHope that helps!'
print("Messy JSON: ", safe_parse_json(messy))

# Test case 4: Totally broken — no JSON at all
broken = "I'm sorry, I can't extract that information."
print("Broken:     ", safe_parse_json(broken))

The function handles cases 1-3 and returns `None` for case 4, which we can use to trigger a retry.

---

## 4. Schema Validation

Parsing succeeded — we have a Python dict. But is it the *right* dict? Does it have the keys we expected, with the types we need?

A simple schema validator checks two things:
1. **Required keys are present**
2. **Values have the expected types**

This catches the common LLM failure mode of returning valid JSON that doesn't match your schema (wrong key names, strings instead of numbers, etc.).

In [ ]:
def validate_schema(data, schema):
    """Validate parsed JSON against an expected schema.

    schema format: {"key": type, ...} where type is str, int, float, list, etc.
    Returns list of error strings (empty = valid).
    """
    errors = []
    for key, expected_type in schema.items():
        if key not in data:
            errors.append(f"Missing required key: '{key}'")
        elif not isinstance(data[key], expected_type):
            errors.append(
                f"Key '{key}' should be {expected_type.__name__}, "
                f"got {type(data[key]).__name__}"
            )
    return errors

In [ ]:
person_schema = {"name": str, "age": int, "occupation": str}

# Good data
good = {"name": "Marie Curie", "age": 159, "occupation": "physicist"}
print("Good data errors:", validate_schema(good, person_schema))

# Missing key
missing = {"name": "Marie Curie", "age": 159}
print("Missing key errors:", validate_schema(missing, person_schema))

# Wrong type — age is a string instead of int
wrong_type = {"name": "Marie Curie", "age": "159", "occupation": "physicist"}
print("Wrong type errors:", validate_schema(wrong_type, person_schema))

# Multiple issues
bad = {"nombre": "Marie Curie"}
print("Multiple errors: ", validate_schema(bad, person_schema))

The validator returns a list of error strings. An empty list means the data is valid. When there *are* errors, we can feed them back to the LLM as part of a retry — which is exactly what the next section does.

---

## 5. Retry Logic

Here's the key insight: **the LLM can fix its own mistakes if you tell it what went wrong**.

When parsing or validation fails, we:
1. Append the LLM's failed response to the conversation history (as an `assistant` message)
2. Append an error message explaining what went wrong (as a `user` message)
3. Call the LLM again — it now has context about its mistake

This is a simple but powerful pattern. The LLM sees its own output, sees the error, and almost always corrects itself on the next attempt.

In [ ]:
import time


def extract_json(prompt, schema=None, max_retries=3, temperature=0):
    """Extract JSON from LLM with retry on failure.

    Args:
        prompt: The user prompt requesting JSON output.
        schema: Optional dict of {key: type} for validation.
        max_retries: Maximum number of retry attempts.
        temperature: LLM sampling temperature.

    Returns:
        Parsed dict on success, None after all retries exhausted.
    """
    messages = [{"role": "user", "content": prompt}]

    for attempt in range(max_retries + 1):
        try:
            response = chat(messages, temperature=temperature)
        except Exception as e:
            print(f"  Attempt {attempt + 1}: API error — {e}")
            time.sleep(1)
            continue

        print(f"  Attempt {attempt + 1}: {response[:100]}...")

        # Try to parse
        parsed = safe_parse_json(response)

        if parsed is None:
            error_msg = (
                "Your response was not valid JSON. "
                "Please respond with ONLY valid JSON, no other text."
            )
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": error_msg})
            time.sleep(1)
            continue

        # If we have a schema, validate
        if schema:
            errors = validate_schema(parsed, schema)
            if errors:
                error_msg = (
                    f"JSON was valid but didn't match the expected schema. "
                    f"Errors: {errors}. "
                    f"Please fix and respond with ONLY the corrected JSON."
                )
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": error_msg})
                time.sleep(1)
                continue

        return parsed

    print(f"  Failed after {max_retries + 1} attempts.")
    return None

Let's test it. With `temperature=0` and a clear prompt, it should succeed on the first attempt. But the retry machinery is there if it doesn't.

In [ ]:
prompt = """Extract the following from the text and respond ONLY with valid JSON:
{
  "name": "person's name (string)",
  "age": "their current age or age at death (integer)",
  "occupation": "their primary job (string)"
}

Text: Albert Einstein was a theoretical physicist born in 1879. He developed the theory of relativity and won the Nobel Prize in Physics in 1921. He died in 1955 at age 76."""

result = extract_json(
    prompt,
    schema={"name": str, "age": int, "occupation": str},
)
print(f"\nResult: {result}")

### How the retry conversation looks

To understand what happens when the LLM *does* fail, here's what the message history looks like after a retry:

```
Turn 1 (user):    "Extract the following... Respond ONLY with valid JSON..."
Turn 2 (assistant): "Sure! Here's the JSON: {\"name\": ...}"     <-- has extra text
Turn 3 (user):    "Your response was not valid JSON. Please respond with ONLY valid JSON."
Turn 4 (assistant): "{\"name\": \"Albert Einstein\", ...}"       <-- fixed!
```

The LLM sees its own mistake and the correction request, which almost always produces clean output on the second attempt.

---

## 6. Extraction Pipeline

Now let's wrap everything into a clean, high-level function. The user just provides text and a description of what to extract — the function handles prompt construction, parsing, validation, and retry internally.

This is the kind of utility you'd put in `utils/` and import across multiple agent notebooks.

In [ ]:
def extract_structured(text, fields, model=None):
    """Extract structured data from text using an LLM.

    Args:
        text: The text to extract information from.
        fields: Dict of {field_name: description} describing what to extract.
        model: Optional model override.

    Returns:
        Parsed dict on success, None on failure.
    """
    schema_desc = json.dumps(
        {k: f"<{v}>" for k, v in fields.items()}, indent=2
    )

    prompt = f"""Extract the following information from the text below.
Respond ONLY with a valid JSON object matching this schema:
{schema_desc}

Text: {text}"""

    return extract_json(prompt, temperature=0)

Let's use it to extract information from a paragraph about a historical figure:

In [ ]:
paragraph = (
    "Ada Lovelace was born in London in 1815. She is widely regarded as the "
    "first computer programmer, having written an algorithm for Charles Babbage's "
    "Analytical Engine. She was the daughter of the poet Lord Byron. She died "
    "in 1852 at the age of 36."
)

fields = {
    "name": "person's full name (string)",
    "birth_year": "year of birth (integer)",
    "birthplace": "city of birth (string)",
    "known_for": "what they are known for (string)",
    "death_year": "year of death (integer)",
}

result = extract_structured(paragraph, fields)
print(f"\nExtracted data:")
print(json.dumps(result, indent=2))

In [ ]:
# Another example — different domain, same function
product_text = (
    "The Sony WH-1000XM5 wireless headphones retail for $349.99. They feature "
    "active noise cancellation, 30-hour battery life, and Bluetooth 5.2. "
    "Released in May 2022, they weigh 250 grams."
)

product_fields = {
    "product_name": "full product name (string)",
    "brand": "manufacturer (string)",
    "price": "price in dollars (number)",
    "features": "list of key features (list of strings)",
    "release_year": "year released (integer)",
}

result = extract_structured(product_text, product_fields)
print("\nExtracted data:")
print(json.dumps(result, indent=2))

Notice how the same `extract_structured` function works across completely different domains. The LLM does the hard work of understanding the text; our code handles the reliable extraction of structure.

---

## Putting It Together: Entity Extraction Agent

Let's combine everything into a realistic end-to-end pipeline. Given a news-style paragraph, we'll extract all the key entities (people, places, dates, topics) as structured JSON, with full retry and validation.

This is a simplified version of what real information-extraction agents do.

In [ ]:
def extract_entities(text, max_retries=3):
    """Extract named entities from text as structured JSON.

    Returns a dict with keys: people, places, dates, topics.
    Each value is a list of strings.
    """
    prompt = f"""Analyze the following text and extract all named entities.
Respond ONLY with a valid JSON object using this exact schema:
{{
  "people": ["list of person names mentioned"],
  "places": ["list of locations mentioned"],
  "dates": ["list of dates or time references mentioned"],
  "topics": ["list of key topics or subjects discussed"]
}}

Text: {text}"""

    schema = {"people": list, "places": list, "dates": list, "topics": list}
    return extract_json(prompt, schema=schema, max_retries=max_retries)

In [ ]:
article = (
    "On March 15, 2024, researchers at DeepMind published a paper demonstrating "
    "that their new Gemini model could outperform GPT-4 on several benchmarks. "
    "The work, led by Demis Hassabis in London, focused on mathematical reasoning "
    "and code generation tasks. Google CEO Sundar Pichai announced the results "
    "at a press conference in Mountain View, California, calling it a "
    "'significant milestone in AI research.'"
)

print("Input text:")
print(article)
print("\n" + "=" * 60)
print("Extracting entities...\n")

entities = extract_entities(article)

if entities:
    print("\nExtracted entities:")
    print(json.dumps(entities, indent=2))
    print(f"\nTotal entities found: {sum(len(v) for v in entities.values())}")
else:
    print("\nFailed to extract entities after all retries.")

Let's also see the full pipeline in action with a second example to show it generalizes:

In [ ]:
second_article = (
    "In January 2023, the European Union passed the AI Act in Brussels, "
    "making it the world's first comprehensive AI regulation. EU Commissioner "
    "Thierry Breton and Parliament member Dragos Tudorache were key architects "
    "of the legislation. The act covers facial recognition, generative AI, and "
    "high-risk AI systems used in healthcare and law enforcement."
)

print("Extracting entities from second article...\n")
entities_2 = extract_entities(second_article)

if entities_2:
    print("\nExtracted entities:")
    print(json.dumps(entities_2, indent=2))
else:
    print("\nExtraction failed.")

### What we built

The full pipeline for this notebook, from bottom to top:

```
extract_entities()          <-- domain-specific function
    └── extract_json()      <-- retry loop with error feedback
        ├── chat()          <-- LLM API call
        ├── safe_parse_json()  <-- JSON extraction from messy text
        └── validate_schema()  <-- schema type checking
```

Each layer is independent and reusable. You can swap out the LLM, change the schema, or adjust the retry strategy without touching the other pieces.

---

## Try It Yourself

### Exercise 1: Movie Review Extraction

Use `extract_json()` (or build on `extract_structured()`) to extract structured data from a movie review. Your output should match this schema:

```python
{"title": str, "rating": (int or float), "sentiment": str, "summary": str}
```

Test with this review:
```
"I just watched Oppenheimer and it was absolutely phenomenal. Christopher Nolan 
outdid himself with the cinematography and Cillian Murphy's performance was 
Oscar-worthy. I'd give it a 9 out of 10. The only downside was the runtime — 
at 3 hours, it felt a bit long in the middle."
```

In [ ]:
# Exercise 1: Your code here
review = (
    "I just watched Oppenheimer and it was absolutely phenomenal. Christopher Nolan "
    "outdid himself with the cinematography and Cillian Murphy's performance was "
    "Oscar-worthy. I'd give it a 9 out of 10. The only downside was the runtime — "
    "at 3 hours, it felt a bit long in the middle."
)


### Exercise 2: Better Error Handling

Modify `extract_json()` to return a clear error dict `{"error": "...", "attempts": N}` instead of `None` after exhausting retries. The error message should include what went wrong on the last attempt (parse failure vs. schema mismatch).

Also add a `max_retries` parameter to `extract_structured()` and pass it through.

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Nested Schema Validation

Extend `validate_schema()` to handle nested objects and lists. For example:

```python
schema = {
    "name": str,
    "address": {"city": str, "country": str},      # nested object
    "skills": [str],                                  # list of strings
}
```

Your enhanced validator should check:
- Nested dicts have the right keys and types
- Lists contain elements of the specified type
- Still returns a list of error strings

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Chained Extraction

Build a two-step pipeline:

1. **Step 1:** Extract entities from a text (people, places, topics) using `extract_entities()`
2. **Step 2:** Use the extracted entities to generate follow-up questions, also as structured JSON:
   ```python
   {"questions": ["question 1", "question 2", ...]}
   ```

For example, if Step 1 extracts `{"people": ["Marie Curie"], "topics": ["radioactivity"]}`, Step 2 should generate questions like `["What awards did Marie Curie win?", "How was radioactivity discovered?"]`.

Both steps must return structured JSON — use the retry loop for both.

In [ ]:
# Exercise 4: Your code here


---

## Summary

In this notebook you built a complete structured-output pipeline:

| Layer | Function | What it does |
|-------|----------|-------------|
| **Prompting** | schema in prompt | Tells the LLM exactly what JSON shape to return |
| **Parsing** | `safe_parse_json()` | Extracts JSON from code fences, surrounding text, or clean output |
| **Validation** | `validate_schema()` | Checks that parsed data has the right keys and types |
| **Retry** | `extract_json()` | Feeds errors back to the LLM for self-correction |
| **Pipeline** | `extract_structured()` | Clean high-level API combining all layers |

These patterns are fundamental to building reliable agents. In the next notebook, you'll use structured output to implement the **ReAct** (Reasoning + Acting) pattern, where the LLM returns structured thought-action-observation steps.

**Next up:** [05. ReAct Agent](05_react_agent.ipynb) — structured reasoning loops with Thought / Action / Observation